# YOLO26 Tri-Dataset Training & Verification Pipeline (v1 — Non-Destructive)

**AI Powered Crop Intelligence Rover · Official Ultralytics YOLO26 Upgrade**

### Strict Engineering & Scientific Guarantees
1. **Strictly Non-Destructive**: Never modifies `/kaggle/input/*`, existing `cv/models/*` (`yolo_plantdoc_best.pt`, `yolov8n_lesions_best.pt`, `efficientnetv2_s_best.pt`, `mobile_unet_best.pt`), or existing notebooks. All outputs write exclusively to `artifacts/yolo26_v1/` and `runs/yolo26_v1/`.
2. **Task Separation**:
   - **Task A (Whole-Leaf Specimen Boundary Detection)**: Official `YOLO26` (`yolo26n.pt` / `yolo26s.pt` / `yolo26m.pt`) trained solely on genuine **PlantDoc** Pascal VOC bounding boxes in canonical alphabetical 29-class order.
   - **Task B (Granular Lesion Segmentation)**: Official `YOLO26-Seg` (`yolo26n-seg.pt` / `yolo26s-seg.pt`) trained solely on genuine human-annotated **PlantSeg** masks/polygons when mounted. Explicitly blocks HSV/LAB pseudo-annotations (`yolo_lesions`, `cv/datasets/processed/segmentation`).
   - **Classification-Only Datasets (`PlantVillage`, `PlantWild`)**: Never converted into synthetic bounding boxes; audited with exact licenses (**PlantWild: `CC BY-NC-ND 4.0`** non-commercial no-derivatives restriction) and used solely for healthy negative-control false-positive evaluation and auxiliary robustness checks.
3. **Genuine Architecture Verification**: Confirms `end2end=True`, `reg_max=1`, `one2one`, and `one2many` heads in Ultralytics `>= 8.4` and benchmarks both **NMS-free (`end2end=True`)** and **one-to-many (`end2end=False`)** modes against preserved `YOLO11n` and `YOLOv8n` baselines on identical untouched test images.


In [ ]:
# ==============================================================================
# Cell 0: Hardware Gatekeeper, CUDA/VRAM Audit & Single-GPU Policy (Non-Destructive)
# ==============================================================================
import os
import sys
import gc
import json
import time
import shutil
import random
import platform
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path(".").resolve()

ARTIFACTS_DIR = WORK_ROOT / "artifacts" / "yolo26_v1"
RUNS_DIR = WORK_ROOT / "runs" / "yolo26_v1"
DATASETS_OUT_DIR = ARTIFACTS_DIR / "datasets"
WEIGHTS_OUT_DIR = ARTIFACTS_DIR / "checkpoints"
EVAL_OUT_DIR = ARTIFACTS_DIR / "evaluation"

for d in (ARTIFACTS_DIR, RUNS_DIR, DATASETS_OUT_DIR, WEIGHTS_OUT_DIR, EVAL_OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Single-GPU policy prevents DataParallel host-RAM leaks on Kaggle T4 x2 / P100
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
DEVICE_STR = "0" if NUM_GPUS > 0 else ("mps" if torch.backends.mps.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if NUM_GPUS > 0 else DEVICE_STR.upper()
VRAM_GB = (torch.cuda.get_device_properties(0).total_memory / (1024**3)) if NUM_GPUS > 0 else 0.0
CAPABILITY = torch.cuda.get_device_capability(0) if NUM_GPUS > 0 else (0, 0)
SM_TAG = f"sm_{CAPABILITY[0]}{CAPABILITY[1]}" if NUM_GPUS > 0 else "N/A"
USE_AMP = bool(NUM_GPUS > 0 and CAPABILITY >= (6, 0))

disk_usage = shutil.disk_usage(WORK_ROOT)
FREE_DISK_GB = disk_usage.free / (1024**3)

print("=" * 84)
print("YOLO26 TRI-DATASET TRAINING v1 — HARDWARE PREFLIGHT & SINGLE-GPU GATEKEEPER")
print("=" * 84)
print(f"  Environment      : {'Kaggle' if IS_KAGGLE else 'Local'} ({platform.platform()})")
print(f"  PyTorch / CUDA   : {torch.__version__} | CUDA {torch.version.cuda or 'None'}")
print(f"  Primary Device   : {GPU_NAME} ({SM_TAG}) | Total GPUs visible: {NUM_GPUS}")
print(f"  Training Policy  : Single-GPU device='{DEVICE_STR}' | AMP={USE_AMP} | VRAM={VRAM_GB:.2f} GB")
print(f"  Free Disk Space  : {FREE_DISK_GB:.2f} GB at {WORK_ROOT}")
print(f"  Versioned Output : {ARTIFACTS_DIR}")
print("=" * 84)

if IS_KAGGLE and NUM_GPUS == 0:
    print("WARNING: Kaggle GPU accelerator is currently OFF. Enable GPU T4 or P100 before full training.")


In [ ]:
# ==============================================================================
# Cell 1: Dependency Verification & Genuine Ultralytics YOLO26 Architecture Check
# ==============================================================================
import subprocess
import importlib
import hashlib

MIN_ULTRALYTICS = (8, 4, 0)

def _ver_tuple(v_str: str):
    parts = []
    for tok in str(v_str).split(".")[:3]:
        digits = "".join(ch for ch in tok if ch.isdigit())
        parts.append(int(digits or 0))
    return tuple((parts + [0, 0, 0])[:3])

try:
    import ultralytics
    if _ver_tuple(ultralytics.__version__) < MIN_ULTRALYTICS:
        raise ImportError(f"Ultralytics {ultralytics.__version__} < 8.4.0 (YOLO26 requires >= 8.4)")
except ImportError as exc:
    print(f"[*] Provisioning ultralytics>=8.4.0 ({exc})...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics>=8.4.0"], check=True)
    ultralytics = importlib.reload(importlib.import_module("ultralytics"))

import cv2
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from ultralytics import YOLO

def verify_yolo26_checkpoint_arch(model_name_or_path: str) -> dict:
    model = YOLO(model_name_or_path)
    nn_mod = model.model
    yaml_info = getattr(nn_mod, "yaml", None) or {}
    yaml_file = str(yaml_info.get("yaml_file", "")).lower()
    head = nn_mod.model[-1]
    is_e2e = bool(getattr(head, "end2end", False))
    reg_max = getattr(head, "reg_max", None)
    has_o2m = hasattr(head, "one2many")
    has_o2o = hasattr(head, "one2one")
    params = sum(p.numel() for p in nn_mod.parameters())
    assert "yolo26" in yaml_file and is_e2e and has_o2m and has_o2o, (
        f"Architecture verification failed for {model_name_or_path}: "
        f"yaml_file={yaml_file}, end2end={is_e2e}, one2many={has_o2m}, one2one={has_o2o}"
    )
    return {
        "model_ref": model_name_or_path,
        "yaml_file": yaml_file,
        "head_class": type(head).__name__,
        "end2end": is_e2e,
        "reg_max": reg_max,
        "one2many_head": has_o2m,
        "one2one_head": has_o2o,
        "parameters": int(params),
    }

YOLO26_VARIANTS_CHECK = {}
for cfg in ("yolo26n.yaml", "yolo26s.yaml", "yolo26m.yaml", "yolo26n-seg.yaml", "yolo26s-seg.yaml"):
    YOLO26_VARIANTS_CHECK[cfg] = verify_yolo26_checkpoint_arch(cfg)

ENV_PROVENANCE = {
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "opencv": cv2.__version__,
    "numpy": np.__version__,
    "gpu": GPU_NAME,
    "vram_gb": round(VRAM_GB, 2),
    "yolo26_architectures_verified": YOLO26_VARIANTS_CHECK,
}
print(json.dumps(ENV_PROVENANCE, indent=2))


In [ ]:
# ==============================================================================
# Cell 2: Kaggle Input Discovery, License Governance & Existing Baseline Audit
# ==============================================================================
SEARCH_ROOTS = []
for cand in [Path("/kaggle/input"), Path("."), Path("..")]:
    if cand.exists() and cand not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(cand)

# Import non-destructive project preparation/evaluation modules if mounted
CODEBASE_ROOT = None
for r in SEARCH_ROOTS:
    for p in r.rglob("prepare_yolo26_tridataset.py"):
        CODEBASE_ROOT = p.parents[2]
        break
    if CODEBASE_ROOT is not None:
        break

if CODEBASE_ROOT is not None:
    if str(CODEBASE_ROOT / "cv" / "scripts") not in sys.path:
        sys.path.insert(0, str(CODEBASE_ROOT / "cv" / "scripts"))
    print(f"✓ Located codebase scripts at: {CODEBASE_ROOT}")
else:
    print("[*] prepare_yolo26_tridataset.py not in mounted codebase; inspect paths manually before proceeding.")

# Locate preserved production baselines (READ-ONLY)
def find_readonly_artifact(filename: str):
    for r in SEARCH_ROOTS + [WORK_ROOT]:
        if not r.exists():
            continue
        for match in r.rglob(filename):
            if "artifacts/yolo26_v1" in str(match):
                continue
            if match.is_file() and match.stat().st_size > 1024:
                return match
    return None

LEGACY_BASELINES = {
    "specimen_yolo11n": find_readonly_artifact("yolo_plantdoc_best.pt"),
    "lesion_yolov8n": find_readonly_artifact("yolov8n_lesions_best.pt"),
    "classifier_effnetv2s": find_readonly_artifact("efficientnetv2_s_best.pt"),
    "segmenter_mobile_unet": find_readonly_artifact("mobile_unet_best.pt"),
}
print("Preserved Production Checkpoints (READ-ONLY):")
for k, p in LEGACY_BASELINES.items():
    print(f"  • {k:<22}: {p}")

# Document Tri-Dataset Licenses & Restrictions cleanly before touching data
LICENSE_AUDIT = {
    "PlantVillage": "CC0 1.0 Universal (Hughes & Salathe 2015) — Classification-only, no spatial boxes.",
    "PlantDoc": "MIT License (Singh et al. 2019) — Genuine 29-class whole-leaf Pascal VOC bounding boxes.",
    "PlantWild": "RESTRICTED: CC BY-NC-ND 4.0 (Wei et al. 2024) — Non-commercial research only, no derivative redistribution. Classification-only.",
    "PlantSeg": "Academic/Zenodo Research Access (Wei et al. 2024) — Verify repository LICENSE before redistribution. Genuine pixel-level disease masks.",
}
print("\nDataset License Governance:")
for k, v in LICENSE_AUDIT.items():
    print(f"  • {k:<14}: {v}")


In [ ]:
# ==============================================================================
# Cell 3: Leakage-Free Dataset Preparation, Class Imbalance & Visual GT Audit
# ==============================================================================
from prepare_yolo26_tridataset import run_preparation, DATASET_LICENSES

PREP_REPORT = run_preparation(SEARCH_ROOTS, DATASETS_OUT_DIR, seed=SEED)
print("Dataset Preparation Summary:")
print(json.dumps({
    "discovered_paths": PREP_REPORT["discovered_paths"],
    "plantdoc_specimen_detection": {
        k: PREP_REPORT["plantdoc_specimen_detection"][k]
        for k in ("status", "splits", "boxes", "small_boxes_under_1pct", "cross_split_duplicates_removed", "intra_split_duplicates_removed", "invalid_boxes_skipped")
    },
    "plantseg_lesion_segmentation": PREP_REPORT["plantseg_lesion_segmentation"],
    "healthy_negative_controls": PREP_REPORT["healthy_negative_controls"],
}, indent=2))

PLANTDOC_YAML = PREP_REPORT["plantdoc_specimen_detection"].get("yaml_path")
PLANTSEG_YAML = PREP_REPORT["plantseg_lesion_segmentation"].get("yaml_path")

# Visual ground-truth box verification (confirms coordinate & label alignment before training)
if PLANTDOC_YAML and Path(PLANTDOC_YAML).exists():
    sample_imgs = sorted((DATASETS_OUT_DIR / "plantdoc_yolo26" / "images" / "train").glob("*.*"))[:4]
    if sample_imgs:
        fig, axes = plt.subplots(1, len(sample_imgs), figsize=(4 * len(sample_imgs), 4))
        if len(sample_imgs) == 1:
            axes = [axes]
        import yaml
        pd_names = yaml.safe_load(Path(PLANTDOC_YAML).read_text())["names"]
        for ax, ip in zip(axes, sample_imgs):
            im = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
            ih, iw = im.shape[:2]
            lp = DATASETS_OUT_DIR / "plantdoc_yolo26" / "labels" / "train" / f"{ip.stem}.txt"
            if lp.exists():
                for line in lp.read_text().splitlines():
                    cid, xc, yc, bw, bh = map(float, line.split()[:5])
                    x1, y1 = int((xc - bw/2)*iw), int((yc - bh/2)*ih)
                    x2, y2 = int((xc + bw/2)*iw), int((yc + bh/2)*ih)
                    cv2.rectangle(im, (x1, y1), (x2, y2), (82, 183, 136), 2)
                    cv2.putText(im, pd_names[int(cid)], (x1, max(14, y1-4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (82, 183, 136), 1)
            ax.imshow(im)
            ax.set_title(ip.name[:24], fontsize=9)
            ax.axis("off")
        plt.tight_layout()
        plt.savefig(EVAL_OUT_DIR / "gt_plantdoc_verification.png", dpi=140)
        plt.show()


In [ ]:
# ==============================================================================
# Cell 4: Hardware-Adaptive Resolution, Batching & Resumption Configuration
# ==============================================================================
# Automatically adapts batch size & candidate matrix to P100 (16GB) vs T4 (15GB)
if VRAM_GB >= 20.0:
    SPECIMEN_EXPERIMENTS = [
        {"name": "yolo26n_640", "weights": "yolo26n.pt", "imgsz": 640, "batch": 16, "epochs": 60},
        {"name": "yolo26s_640", "weights": "yolo26s.pt", "imgsz": 640, "batch": 16, "epochs": 80},
        {"name": "yolo26s_768", "weights": "yolo26s.pt", "imgsz": 768, "batch": 8,  "epochs": 80},
        {"name": "yolo26m_640", "weights": "yolo26m.pt", "imgsz": 640, "batch": 8,  "epochs": 60},
    ]
    SEG_EXPERIMENTS = [
        {"name": "yolo26s_seg_768", "weights": "yolo26s-seg.pt", "imgsz": 768, "batch": 8, "epochs": 80},
        {"name": "yolo26s_seg_960", "weights": "yolo26s-seg.pt", "imgsz": 960, "batch": 4, "epochs": 60},
    ]
elif VRAM_GB >= 14.0:  # Kaggle P100 16GB / T4 15GB
    SPECIMEN_EXPERIMENTS = [
        {"name": "yolo26n_640", "weights": "yolo26n.pt", "imgsz": 640, "batch": 16, "epochs": 60},
        {"name": "yolo26s_640", "weights": "yolo26s.pt", "imgsz": 640, "batch": 8,  "epochs": 80},
        {"name": "yolo26s_768", "weights": "yolo26s.pt", "imgsz": 768, "batch": 6,  "epochs": 80},
    ]
    SEG_EXPERIMENTS = [
        {"name": "yolo26n_seg_640", "weights": "yolo26n-seg.pt", "imgsz": 640, "batch": 8, "epochs": 70},
        {"name": "yolo26s_seg_768", "weights": "yolo26s-seg.pt", "imgsz": 768, "batch": 4, "epochs": 80},
    ]
else:
    SPECIMEN_EXPERIMENTS = [
        {"name": "yolo26n_640", "weights": "yolo26n.pt", "imgsz": 640, "batch": 4, "epochs": 50},
    ]
    SEG_EXPERIMENTS = [
        {"name": "yolo26n_seg_640", "weights": "yolo26n-seg.pt", "imgsz": 640, "batch": 4, "epochs": 50},
    ]

print("Planned Specimen Experiments:", json.dumps(SPECIMEN_EXPERIMENTS, indent=2))
print("Planned Lesion Seg Experiments:", json.dumps(SEG_EXPERIMENTS, indent=2))


In [ ]:
# ==============================================================================
# Cell 5: Task A — Dedicated YOLO26 Specimen Boundary Training (PlantDoc)
# ==============================================================================
SPECIMEN_RUN_RESULTS = []
BEST_SPECIMEN_CKPT = None

if PLANTDOC_YAML and Path(PLANTDOC_YAML).exists() and NUM_GPUS > 0:
    for exp in SPECIMEN_EXPERIMENTS:
        run_dir = RUNS_DIR / "specimen" / exp["name"]
        last_ckpt = run_dir / "weights" / "last.pt"
        best_ckpt = run_dir / "weights" / "best.pt"
        resume_flag = bool(last_ckpt.exists() and not best_ckpt.exists())

        print(f"\n>>> Running Task A Specimen Experiment: {exp['name']} (resume={resume_flag})")
        model = YOLO(str(last_ckpt) if resume_flag else exp["weights"])
        model.train(
            data=str(PLANTDOC_YAML),
            epochs=exp["epochs"],
            imgsz=exp["imgsz"],
            batch=exp["batch"],
            device=DEVICE_STR,
            workers=2,
            amp=USE_AMP,
            cos_lr=True,
            patience=15,
            save=True,
            save_period=10,
            resume=resume_flag,
            project=str(RUNS_DIR / "specimen"),
            name=exp["name"],
            exist_ok=True,
            # Field-realistic augmentations
            hsv_h=0.02,
            hsv_s=0.65,
            hsv_v=0.45,
            degrees=15.0,
            translate=0.12,
            scale=0.50,
            shear=2.0,
            fliplr=0.50,
            flipud=0.15,
            mosaic=0.85,
            mixup=0.10,
            close_mosaic=12,
        )

        # Evaluate trained best.pt strictly on held-out val split
        best_model = YOLO(str(best_ckpt))
        val_res = best_model.val(data=str(PLANTDOC_YAML), imgsz=exp["imgsz"], split="val", verbose=False)
        res_entry = {
            "name": exp["name"],
            "weights": str(best_ckpt),
            "imgsz": exp["imgsz"],
            "map50": round(float(val_res.box.map50), 4),
            "map50_95": round(float(val_res.box.map), 4),
            "precision": round(float(val_res.box.mp), 4),
            "recall": round(float(val_res.box.mr), 4),
        }
        SPECIMEN_RUN_RESULTS.append(res_entry)
        print("Validation Metrics:", res_entry)

        del model, best_model
        torch.cuda.empty_cache()
        gc.collect()

    if SPECIMEN_RUN_RESULTS:
        winner = max(SPECIMEN_RUN_RESULTS, key=lambda r: (r["map50_95"], r["map50"]))
        BEST_SPECIMEN_CKPT = WEIGHTS_OUT_DIR / "yolo26_specimen_best.pt"
        shutil.copy2(winner["weights"], BEST_SPECIMEN_CKPT)
        print(f"\n✓ Selected Best Specimen Checkpoint: {winner['name']} -> {BEST_SPECIMEN_CKPT}")
else:
    print("Task A training skipped locally (requires Kaggle GPU + verified PlantDoc dataset).")


In [ ]:
# ==============================================================================
# Cell 6: Task B — Dedicated YOLO26-Seg Lesion Localization (Genuine PlantSeg Only)
# ==============================================================================
LESION_RUN_RESULTS = []
BEST_LESION_CKPT = None

if PLANTSEG_YAML and Path(PLANTSEG_YAML).exists() and NUM_GPUS > 0:
    for exp in SEG_EXPERIMENTS:
        run_dir = RUNS_DIR / "lesion_seg" / exp["name"]
        last_ckpt = run_dir / "weights" / "last.pt"
        best_ckpt = run_dir / "weights" / "best.pt"
        resume_flag = bool(last_ckpt.exists() and not best_ckpt.exists())

        print(f"\n>>> Running Task B Lesion Segmentation: {exp['name']} (resume={resume_flag})")
        model = YOLO(str(last_ckpt) if resume_flag else exp["weights"])
        model.train(
            data=str(PLANTSEG_YAML),
            epochs=exp["epochs"],
            imgsz=exp["imgsz"],
            batch=exp["batch"],
            device=DEVICE_STR,
            workers=2,
            amp=USE_AMP,
            cos_lr=True,
            patience=18,
            save=True,
            save_period=10,
            resume=resume_flag,
            project=str(RUNS_DIR / "lesion_seg"),
            name=exp["name"],
            exist_ok=True,
            # Small-object lesion optimization
            overlap_mask=True,
            mask_ratio=2,
            copy_paste=0.25,
            mosaic=0.80,
            close_mosaic=15,
            hsv_h=0.015,
            hsv_s=0.55,
            hsv_v=0.40,
            scale=0.55,
            fliplr=0.50,
        )

        best_model = YOLO(str(best_ckpt))
        val_res = best_model.val(data=str(PLANTSEG_YAML), imgsz=exp["imgsz"], split="val", verbose=False)
        res_entry = {
            "name": exp["name"],
            "weights": str(best_ckpt),
            "imgsz": exp["imgsz"],
            "box_map50": round(float(val_res.box.map50), 4),
            "box_map50_95": round(float(val_res.box.map), 4),
            "mask_map50": round(float(val_res.seg.map50), 4) if hasattr(val_res, "seg") else None,
            "mask_map50_95": round(float(val_res.seg.map), 4) if hasattr(val_res, "seg") else None,
        }
        LESION_RUN_RESULTS.append(res_entry)
        print("Validation Metrics:", res_entry)

        del model, best_model
        torch.cuda.empty_cache()
        gc.collect()

    if LESION_RUN_RESULTS:
        winner = max(LESION_RUN_RESULTS, key=lambda r: (r["mask_map50_95"] or 0.0, r["box_map50"]))
        BEST_LESION_CKPT = WEIGHTS_OUT_DIR / "yolo26_lesion_best.pt"
        shutil.copy2(winner["weights"], BEST_LESION_CKPT)
        print(f"\n✓ Selected Best Lesion Checkpoint: {winner['name']} -> {BEST_LESION_CKPT}")
else:
    print("Task B Lesion Segmentation SKIPPED HONESTLY: Genuine PlantSeg masks not found (no HSV pseudo-fallback).")


In [ ]:
# ==============================================================================
# Cell 7: Independent Held-Out Evaluation (end2end vs one2many & vs YOLO11n/v8n)
# ==============================================================================
from evaluate_yolo26_v1 import evaluate_all

EVAL_REPORT = evaluate_all(
    out_dir=EVAL_OUT_DIR,
    baseline_specimen=LEGACY_BASELINES["specimen_yolo11n"] or Path("cv/models/yolo_plantdoc_best.pt"),
    baseline_lesion=LEGACY_BASELINES["lesion_yolov8n"] or Path("cv/models/yolov8n_lesions_best.pt"),
    candidate_specimen=BEST_SPECIMEN_CKPT,
    candidate_lesion=BEST_LESION_CKPT,
    plantdoc_test_dir=(DATASETS_OUT_DIR / "plantdoc_yolo26") if PLANTDOC_YAML else None,
    project_root=CODEBASE_ROOT or Path("."),
)
print("Evaluation & Promotion Gate Report:")
print(json.dumps(EVAL_REPORT["promotion_decision"], indent=2))
print(json.dumps(EVAL_REPORT["specimen_comparison"], indent=2))
print(json.dumps(EVAL_REPORT["negative_control_lesion_fp"], indent=2))


In [ ]:
# ==============================================================================
# Cell 8: Versioned Artifact Manifest Export & Download Bundle
# ==============================================================================
from evaluate_yolo26_v1 import sha256_file, inspect_yolo_checkpoint

manifest = {
    "version": "yolo26_v1",
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "environment": ENV_PROVENANCE,
    "dataset_preparation": PREP_REPORT,
    "specimen_experiments": SPECIMEN_RUN_RESULTS,
    "lesion_seg_experiments": LESION_RUN_RESULTS,
    "checkpoints": {
        "yolo26_specimen_best": inspect_yolo_checkpoint(BEST_SPECIMEN_CKPT) if BEST_SPECIMEN_CKPT else None,
        "yolo26_lesion_best": inspect_yolo_checkpoint(BEST_LESION_CKPT) if BEST_LESION_CKPT else None,
    },
    "evaluation": EVAL_REPORT,
    "preserved_production_defaults": {
        "classifier": "cv/models/efficientnetv2_s_best.pt",
        "specimen_detector": "cv/models/yolo_plantdoc_best.pt",
        "lesion_detector": "cv/models/yolov8n_lesions_best.pt",
        "segmenter": "cv/models/mobile_unet_best.pt",
    },
}

manifest_path = ARTIFACTS_DIR / "RELEASE_MANIFEST_YOLO26_v1.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

archive_base = WORK_ROOT / "yolo26_v1_artifacts"
shutil.make_archive(str(archive_base), "zip", root_dir=str(ARTIFACTS_DIR))
print(f"✓ Saved RELEASE_MANIFEST_YOLO26_v1.json: {manifest_path}")
print(f"✓ Downloadable archive ready: {archive_base}.zip")
